# 02 — Prevendo se o cliente recomendaria o produto

**Tarefa:** classificação binária a partir do texto da avaliação (título + comentário).
**Alvo:** `recomenda` (1 = recomendaria a um amigo).

Roteiro: baseline ingênuo → TF-IDF com modelos lineares → escolha do limiar de decisão →
análise de erros → interpretação dos pesos.

**Entrada:** `avaliacoes_preparadas.csv`, gerado pelo `01_eda_texto.ipynb`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score,
                             f1_score, precision_score, recall_score)

pd.set_option("display.max_colwidth", 110)
sns.set_theme(style="whitegrid", context="notebook")

SEMENTE = 42
PASTA_PROCESSED = Path("../data/processed")
PASTA_FIGURAS = Path("../reports/figures")
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

def salvar(fig, nome):
    fig.savefig(PASTA_FIGURAS / f"{nome}.png", dpi=150, bbox_inches="tight")

base = pd.read_csv(PASTA_PROCESSED / "avaliacoes_preparadas.csv")
print(f"{len(base):,} avaliações | taxa de recomendação: {base['recomenda'].mean():.1%}")
base.head(3)

## 1. Separação treino e teste

Divisão estratificada em 80/20, mantendo a proporção das classes. O conjunto de teste fica intocado
até o fim: nenhuma decisão de modelagem é tomada olhando para ele.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    base["texto_limpo"].fillna(""), base["recomenda"],
    test_size=0.2, stratify=base["recomenda"], random_state=SEMENTE)

# Guardamos os índices para a análise de erros no fim
indices_teste = X_teste.index

print(f"Treino: {len(X_treino):,} | Teste: {len(X_teste):,}")
print(f"Proporção da classe positiva — treino: {y_treino.mean():.1%} | teste: {y_teste.mean():.1%}")

## 2. Modelos

O **baseline** responde sempre a classe majoritária: qualquer modelo útil precisa superá-lo.
Depois, três classificadores lineares sobre TF-IDF, que são o padrão de mercado para texto curto:

- **Regressão logística** — devolve probabilidades e coeficientes interpretáveis
- **Linear SVC** — costuma ir bem em texto esparso de alta dimensão
- **Complement Naive Bayes** — rápido e pensado para classes desbalanceadas

O TF-IDF usa unigramas e bigramas (para capturar expressões como "não recomendo" e "chegou quebrado"),
descartando termos que aparecem em menos de 5 avaliações.

In [ ]:
def criar_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True, strip_accents="unicode")

modelos = {
    "baseline (classe majoritária)": make_pipeline(
        criar_tfidf(), DummyClassifier(strategy="most_frequent")),
    "regressão logística": make_pipeline(
        criar_tfidf(), LogisticRegression(max_iter=2000, class_weight="balanced", C=4)),
    "linear SVC": make_pipeline(
        criar_tfidf(), CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)),
    "complement naive bayes": make_pipeline(criar_tfidf(), ComplementNB()),
}

validacao = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
linhas = []
for nome, modelo in modelos.items():
    resultado = cross_validate(modelo, X_treino, y_treino, cv=validacao,
                               scoring=["roc_auc", "f1", "precision", "recall"], n_jobs=-1)
    linhas.append({
        "modelo": nome,
        "ROC AUC": resultado["test_roc_auc"].mean(),
        "F1": resultado["test_f1"].mean(),
        "precisão": resultado["test_precision"].mean(),
        "recall": resultado["test_recall"].mean(),
    })

comparacao = pd.DataFrame(linhas).set_index("modelo").sort_values("ROC AUC", ascending=False)
comparacao.style.format("{:.3f}").background_gradient(cmap="Greens", axis=None)

As métricas acima são da **classe positiva** (recomenda), que é a majoritária. Como o baseline acerta
quase 80% das vezes só chutando, o F1 dele já é alto: é por isso que ROC AUC e as métricas da classe
minoritária contam mais nesse problema.

## 3. Avaliação no conjunto de teste

In [ ]:
melhor_nome = comparacao.index[0]
modelo = modelos[melhor_nome].fit(X_treino, y_treino)
probabilidades = modelo.predict_proba(X_teste)[:, 1]
previsoes = (probabilidades >= 0.5).astype(int)

print(f"Modelo escolhido: {melhor_nome}\n")
print(classification_report(y_teste, previsoes, target_names=["não recomenda", "recomenda"], digits=3))
print(f"ROC AUC: {roc_auc_score(y_teste, probabilidades):.3f}")
print(f"Average precision (classe positiva): {average_precision_score(y_teste, probabilidades):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

matriz = confusion_matrix(y_teste, previsoes)
sns.heatmap(matriz, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["previsto: não", "previsto: sim"],
            yticklabels=["real: não", "real: sim"])
axes[0].set_title("Matriz de confusão (limiar 0,5)")

fpr, tpr, _ = roc_curve(y_teste, probabilidades)
axes[1].plot(fpr, tpr, color="#2a9d8f", lw=2,
             label=f"AUC = {roc_auc_score(y_teste, probabilidades):.3f}")
axes[1].plot([0, 1], [0, 1], ls="--", color="gray", lw=1)
axes[1].set_title("Curva ROC")
axes[1].set_xlabel("Falsos positivos")
axes[1].set_ylabel("Verdadeiros positivos")
axes[1].legend()

precisao, recall, _ = precision_recall_curve(y_teste, probabilidades)
axes[2].plot(recall, precisao, color="#e76f51", lw=2)
axes[2].axhline(y_teste.mean(), ls="--", color="gray", lw=1, label="taxa base")
axes[2].set_title("Precisão x recall (classe recomenda)")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precisão")
axes[2].legend()

plt.tight_layout()
salvar(fig, "05_metricas_modelo")
plt.show()

## 4. Escolhendo o limiar de decisão

O limiar padrão de 0,5 é uma convenção, não uma escolha técnica. Se o objetivo for **encontrar clientes
insatisfeitos** (a classe minoritária), vale ajustar o corte para capturar mais casos negativos,
aceitando alguns alarmes falsos.

A tabela abaixo mostra o efeito do limiar sobre a detecção de quem **não** recomenda.

In [ ]:
linhas = []
for limiar in np.arange(0.2, 0.85, 0.05):
    negativo_previsto = (probabilidades < limiar).astype(int)
    negativo_real = (y_teste == 0).astype(int)
    linhas.append({
        "limiar": limiar,
        "precisão (não recomenda)": precision_score(negativo_real, negativo_previsto, zero_division=0),
        "recall (não recomenda)": recall_score(negativo_real, negativo_previsto),
        "F1 (não recomenda)": f1_score(negativo_real, negativo_previsto),
        "% marcado como insatisfeito": negativo_previsto.mean(),
    })
limiares = pd.DataFrame(linhas).set_index("limiar")

melhor_limiar = limiares["F1 (não recomenda)"].idxmax()
print(f"Melhor limiar pelo F1 da classe minoritária: {melhor_limiar:.2f}")
limiares.style.format("{:.3f}").background_gradient(subset=["F1 (não recomenda)"], cmap="Greens")

## 5. Análise de erros

Onde o modelo erra diz mais sobre o problema do que a métrica agregada.

In [ ]:
resultado = base.loc[indices_teste, ["texto", "overall_rating", "site_category_lv1", "n_palavras"]].copy()
resultado["real"] = y_teste.values
resultado["probabilidade"] = probabilidades
resultado["previsto"] = previsoes
resultado["acertou"] = resultado["real"] == resultado["previsto"]

por_nota = resultado.groupby("overall_rating").agg(
    avaliacoes=("acertou", "size"), acerto=("acertou", "mean"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(por_nota.index, por_nota["acerto"], color="#264653")
axes[0].set_title("Taxa de acerto por nota dada pelo cliente")
axes[0].set_xlabel("Estrelas")
axes[0].set_ylim(0, 1)

sns.boxplot(data=resultado, x="acertou", y="n_palavras", showfliers=False,
            hue="acertou", palette={True: "#2a9d8f", False: "#e76f51"}, legend=False, ax=axes[1])
axes[1].set_title("Tamanho do texto: acertos x erros")
axes[1].set_xlabel("Acertou")

plt.tight_layout()
salvar(fig, "06_analise_erros")
plt.show()

por_nota.assign(acerto=lambda d: (d["acerto"] * 100).round(1))

O modelo vai bem nos extremos (1 e 5 estrelas) e sofre nas notas intermediárias, exatamente
onde os próprios clientes se dividem entre recomendar ou não. É o teto natural do problema,
já identificado na exploração.

In [ ]:
print("Erros mais confiantes — o modelo previu 'recomenda' e o cliente não recomendou:")
falsos_positivos = resultado[(resultado["real"] == 0) & (resultado["previsto"] == 1)]
display(falsos_positivos.nlargest(5, "probabilidade")[["overall_rating", "probabilidade", "texto"]])

print("\nErros mais confiantes — o modelo previu 'não recomenda' e o cliente recomendou:")
falsos_negativos = resultado[(resultado["real"] == 1) & (resultado["previsto"] == 0)]
display(falsos_negativos.nsmallest(5, "probabilidade")[["overall_rating", "probabilidade", "texto"]])

## 6. O que o modelo aprendeu

Num modelo linear sobre TF-IDF os coeficientes são lidos diretamente: quanto maior o peso,
mais aquele termo empurra a previsão para "recomenda".

In [ ]:
# Se o campeão não for a regressão logística, treinamos uma só para interpretar o vocabulário
modelo_interpretavel = (modelo if melhor_nome == "regressão logística"
                        else modelos["regressão logística"].fit(X_treino, y_treino))

vetorizador = modelo_interpretavel.named_steps["tfidfvectorizer"]
classificador = modelo_interpretavel.named_steps["logisticregression"]
pesos = pd.Series(classificador.coef_[0], index=vetorizador.get_feature_names_out())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (dados, titulo, cor) in zip(axes, [
    (pesos.nlargest(15), "Empurram para RECOMENDA", "#2a9d8f"),
    (pesos.nsmallest(15), "Empurram para NÃO RECOMENDA", "#e76f51"),
]):
    dados = dados.sort_values()
    ax.barh(dados.index, dados.values, color=cor)
    ax.set_title(titulo)
    ax.set_xlabel("Peso no modelo")
plt.tight_layout()
salvar(fig, "07_pesos_modelo")
plt.show()

## 7. Testando com frases novas

In [ ]:
frases = [
    "produto excelente, chegou antes do prazo e funciona perfeitamente",
    "chegou quebrado e até hoje não resolveram o meu problema",
    "o produto é bom mas a entrega atrasou demais",
    "cumpre o que promete pelo preço que custa",
]
teste_manual = pd.DataFrame({
    "frase": frases,
    "prob. de recomendar": modelo.predict_proba(frases)[:, 1],
})
teste_manual.style.format({"prob. de recomendar": "{:.1%}"})

## 8. Conclusões e limitações

- Um pipeline simples (TF-IDF + modelo linear) resolve bem a tarefa e supera com folga o baseline.
- O desempenho cai nas notas intermediárias, onde a própria base é ambígua: o cliente gosta do produto
  e reclama da entrega. Esse é o limite do problema, não do modelo.
- O limiar de decisão deve ser escolhido pelo objetivo de negócio. Para **detectar insatisfeitos**,
  vale abrir mão de precisão em troca de recall.
- Os pesos do modelo linear permitem auditar o que foi aprendido, algo que modelos mais complexos
  não entregam de graça.

**Limitações:** a base é de 2018 e de um único varejista, então o vocabulário pode não se transferir
para outro contexto; textos muito curtos dão pouca informação; e o modelo captura padrões de escrita,
não a experiência real do cliente.

**Evolução natural:** comparar este baseline com embeddings do BERTimbau (BERT em português).
O ganho costuma ser modesto em texto curto e o custo computacional é bem maior — mas medir essa
diferença é justamente o tipo de análise que uma vaga de ciência de dados espera.

In [ ]:
import joblib

joblib.dump(modelo, PASTA_PROCESSED / "modelo_recomendacao.joblib")
resultado.to_csv(PASTA_PROCESSED / "previsoes_teste.csv", index=False, encoding="utf-8")
comparacao.to_csv(PASTA_PROCESSED / "comparacao_modelos.csv", encoding="utf-8")
print("Modelo e resultados salvos em", PASTA_PROCESSED)